# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UmairMehfooz/Ml-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [3]:
HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

rel = "hf://datasets/FlyRank/internship-warehouse"

print("DuckDB + Hugging Face connection ready.")

DuckDB + Hugging Face connection ready.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
march_path = (
    f"{rel}/fact_content_daily_performance/"
    "month=2026-03/**/*.parquet"
)

april_path = (
    f"{rel}/fact_content_daily_performance/"
    "month=2026-04/**/*.parquet"
)
march_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(sessions_organic) AS sessions_organic,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
FROM read_parquet('{march_path}')
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
"""

march_features = con.sql(march_query).df()

print("March rows:", len(march_features))
april_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS april_impressions
FROM read_parquet('{april_path}')
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
"""

april_outcome = con.sql(april_query).df()

print("April rows:", len(april_outcome))
model_df = march_features.merge(
    april_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)
model_df["impression_change_pct"] = (
    (
        model_df["april_impressions"]
        - model_df["gsc_impressions"]
    )
    / model_df["gsc_impressions"]
) * 100

model_df["decline_label"] = (
    model_df["impression_change_pct"] < -20
).astype(int)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March rows: 176738


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

April rows: 194760


In [5]:
feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "sessions_organic",
    "ga4_engaged_sessions"
]
model_df = model_df.dropna(
    subset=feature_columns + ["decline_label"]
).reset_index(drop=True)
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        model_df["decline_label"],
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

In [7]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(
    train_df[feature_columns],
    train_df["decline_label"]
)
test_df["decline_risk"] = model.predict_proba(
    test_df[feature_columns]
)[:, 1]

queue = test_df[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "sessions_organic",
        "ga4_engaged_sessions",
        "decline_risk"
    ]
].copy()

queue = queue.sort_values(
    "decline_risk",
    ascending=False
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

queue.head(20)

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,sessions_organic,ga4_engaged_sessions,decline_risk,rank
0,client_fef1a8f436438636,content_c30950c89c368054,467.0,1.0,17.389321,257.0,27.0,0.965406,1
1,client_e5c2aa26a8598242,content_4977e90c4d93cf9f,73862.0,28.0,7.399241,84.0,1.0,0.849155,2
2,client_fef1a8f436438636,content_e3496dac741da4f9,63494.0,155.0,6.500714,370.0,13.0,0.837957,3
3,client_fef1a8f436438636,content_ba462518dad435fc,91391.0,46.0,27.355006,71.0,2.0,0.826848,4
4,client_fef1a8f436438636,content_84a6bf3578312e90,91388.0,85.0,20.839925,130.0,4.0,0.785436,5
5,client_fef1a8f436438636,content_3bfbf4c87870a639,386.0,0.0,18.690467,99.0,10.0,0.765684,6
6,client_fef1a8f436438636,content_7a0a59b4cb181ab9,60164.0,32.0,34.136038,74.0,1.0,0.757334,7
7,client_fef1a8f436438636,content_0aaa197051f58d6f,61071.0,34.0,34.712403,67.0,0.0,0.731817,8
8,client_fef1a8f436438636,content_ec7a08d630336e6c,63.0,0.0,34.284402,90.0,9.0,0.730703,9
9,client_65de48885f4ef01b,content_62673eea26c31c17,57720.0,43.0,6.840768,53.0,6.0,0.719550,10


In [8]:
queue["priority"] = pd.cut(
    queue["decline_risk"],
    bins=[-np.inf, 0.4, 0.6, np.inf],
    labels=["LOW", "MEDIUM", "HIGH"]
)
queue["action"] = np.select(
    [
        queue["priority"] == "HIGH",
        queue["priority"] == "MEDIUM",
        queue["priority"] == "LOW"
    ],
    [
        "Review for potential refresh",
        "Review before deciding on refresh",
        "Monitor; no immediate action"
    ],
    default="Human review"
)

In [9]:
click_threshold = queue["gsc_clicks"].median()
engagement_threshold = queue["ga4_engaged_sessions"].median()
position_threshold = queue["gsc_avg_position"].median()
queue["reason_code"] = np.select(
    [
        queue["gsc_clicks"] <= click_threshold,
        queue["ga4_engaged_sessions"] <= engagement_threshold,
        queue["gsc_avg_position"] >= position_threshold
    ],
    [
        "LOW_SEARCH_CLICKS",
        "LOW_ENGAGEMENT",
        "WEAKER_SEARCH_POSITION"
    ],
    default="COMBINED_SIGNALS"
)
queue.head(20)[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "decline_risk",
        "priority",
        "reason_code",
        "action"
    ]
]

,rank,client_hash_id,content_hash_id,decline_risk,priority,reason_code,action
0,1,client_fef1a8f436438636,content_c30950c89c368054,0.965406,HIGH,LOW_SEARCH_CLICKS,Review for potential refresh
1,2,client_e5c2aa26a8598242,content_4977e90c4d93cf9f,0.849155,HIGH,COMBINED_SIGNALS,Review for potential refresh
2,3,client_fef1a8f436438636,content_e3496dac741da4f9,0.837957,HIGH,COMBINED_SIGNALS,Review for potential refresh
3,4,client_fef1a8f436438636,content_ba462518dad435fc,0.826848,HIGH,WEAKER_SEARCH_POSITION,Review for potential refresh
4,5,client_fef1a8f436438636,content_84a6bf3578312e90,0.785436,HIGH,WEAKER_SEARCH_POSITION,Review for potential refresh
5,6,client_fef1a8f436438636,content_3bfbf4c87870a639,0.765684,HIGH,LOW_SEARCH_CLICKS,Review for potential refresh
6,7,client_fef1a8f436438636,content_7a0a59b4cb181ab9,0.757334,HIGH,WEAKER_SEARCH_POSITION,Review for potential refresh
7,8,client_fef1a8f436438636,content_0aaa197051f58d6f,0.731817,HIGH,LOW_ENGAGEMENT,Review for potential refresh
8,9,client_fef1a8f436438636,content_ec7a08d630336e6c,0.730703,HIGH,LOW_SEARCH_CLICKS,Review for potential refresh
9,10,client_65de48885f4ef01b,content_62673eea26c31c17,0.719550,HIGH,COMBINED_SIGNALS,Review for potential refresh


## 1. Ranked actions + reason codes

The model output is converted into a ranked review queue using estimated decline risk.

Pages are assigned a priority level:

- HIGH: review for potential refresh
- MEDIUM: review before deciding on refresh
- LOW: monitor with no immediate action

Each page also receives one reason code based on the signals used by the model. Example reason codes include `LOW_SEARCH_CLICKS`, `LOW_ENGAGEMENT`, and `WEAKER_SEARCH_POSITION`.

The reason code is intended to explain why an item was surfaced for review. It is not a causal explanation and does not mean that the signal caused the future decline.

The queue is therefore a decision-support tool rather than an automatic publishing or content-editing system.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended use and limits

### Intended use

The intended use is to help a content team prioritize pages for human review.

The model ranks content by estimated decline risk so that reviewers can focus first on pages with stronger model signals.

The output can support decisions such as:

- which pages should be reviewed first;
- which pages may warrant investigation for a content refresh;
- which pages should simply be monitored;
- which signals may explain why a page was prioritized.

### Limits

The model does not determine whether a page should definitely be refreshed.

The ranking is based on historical relationships between the available features and a future impression-decline label.

The model was evaluated on a held-out client-grouped split, but this does not establish performance across all future time periods.

The model also does not include all possible causes of content decline, such as algorithm updates, seasonality, search-intent changes, competitor activity, or changes made to the page.

Therefore the output should be treated as decision support rather than an automatic recommendation.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Human review and no-go list

### Human review rules

Every HIGH or MEDIUM priority item requires human review before action.

The reviewer should check:

1. Whether the page actually has a meaningful decline or risk signal.
2. Whether the search intent has changed.
3. Whether the page is still relevant to the target query.
4. Whether the content is outdated or incomplete.
5. Whether there are known external factors affecting performance.
6. Whether a refresh is appropriate compared with simply monitoring the page.

A model score is one input into the decision, not the decision itself.

### What should NOT be automated

The following actions should not be automated from this model:

- automatically rewriting page content;
- automatically changing titles or meta descriptions;
- automatically publishing content changes;
- automatically deleting pages;
- automatically redirecting URLs;
- automatically declaring a page "bad";
- automatically spending a client's content budget;
- automatically claiming that a refresh caused performance improvement.

These actions require human judgment and additional evidence.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 4. Monitoring and retrain triggers

The model should be monitored rather than assumed to remain valid indefinitely.

### Monitoring signals

I would monitor:

- Precision@50 on newly observed labeled periods.
- Average Precision.
- The proportion of items receiving HIGH priority.
- Feature distributions compared with the training period.
- The distribution of predicted decline risk.
- Changes in the relationship between predictions and observed outcomes.

### Retrain or review triggers

I would investigate retraining or re-validation if:

1. Precision@50 drops materially across multiple future evaluation windows.
2. The distribution of important features changes substantially.
3. The proportion of HIGH-risk recommendations changes unexpectedly.
4. Search behavior or the underlying content environment changes substantially.
5. The label definition or business decision changes.

A single poor month would be treated as a signal for investigation rather than an automatic retraining trigger.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

os.makedirs(
    "work/outputs",
    exist_ok=True
)
export_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "decline_risk",
    "priority",
    "reason_code",
    "action"
]

ranked_queue = queue[export_columns].copy()
output_path = (
    "work/outputs/"
    "action_playbook_queue.csv"
)

ranked_queue.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print(
    ranked_queue.head(10).to_string(index=False)
)

print("\nRows exported:", len(ranked_queue))

Saved: work/outputs/action_playbook_queue.csv
 rank          client_hash_id          content_hash_id  decline_risk priority            reason_code                       action
    1 client_fef1a8f436438636 content_c30950c89c368054      0.965406     HIGH      LOW_SEARCH_CLICKS Review for potential refresh
    2 client_e5c2aa26a8598242 content_4977e90c4d93cf9f      0.849155     HIGH       COMBINED_SIGNALS Review for potential refresh
    3 client_fef1a8f436438636 content_e3496dac741da4f9      0.837957     HIGH       COMBINED_SIGNALS Review for potential refresh
    4 client_fef1a8f436438636 content_ba462518dad435fc      0.826848     HIGH WEAKER_SEARCH_POSITION Review for potential refresh
    5 client_fef1a8f436438636 content_84a6bf3578312e90      0.785436     HIGH WEAKER_SEARCH_POSITION Review for potential refresh
    6 client_fef1a8f436438636 content_3bfbf4c87870a639      0.765684     HIGH      LOW_SEARCH_CLICKS Review for potential refresh
    7 client_fef1a8f436438636 content_7a0a59

In [11]:
import json

summary = {
    "model": "Logistic Regression",
    "features": feature_columns,
    "validation": "client_grouped",
    "w05_precision_at_50": 0.66,
    "average_precision": 0.5903,
    "precision": 0.6309,
    "recall": 0.0271,
    "purpose": "decision-support content prioritization",
    "automatic_content_changes": False
}

with open(
    "work/outputs/w07_playbook_summary.json",
    "w"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print("Summary JSON saved.")

Summary JSON saved.


## 5. Exports for the paper

The notebook exports a ranked content-review queue to:

`work/outputs/action_playbook_queue.csv`

The queue contains the rank, client/content identifiers, estimated decline risk, priority, reason code, and suggested action.

The CSV is regenerated by the notebook and is therefore not treated as a source-controlled data artifact.

A summary JSON records the model, validation design, metrics, and intended use so that the paper can trace the playbook back to the modeling work.

The queue is intended to support human review and is not a production automation system.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.